# Example 1.2.3 / 1: Symbolic Parameters

Companion notebook for **1.2.3 Symbolic and Logical Parameters** in the AMPL lab Documentation, section "Symbolic Parameters". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Symbolic Parameters

With the keyword `symbolic`, a parameter may hold a string. Its values can be strings or
numbers, like a set's members, but string values can't be used in arithmetic.

A common use is to name a set member that the model treats specially, such as the entrance
and exit of a road network. A symbolic value is a Python string: assign it with
`ampl.param["entr"] = "a"`, and `value()` returns it as a string. `is_symbolic()` tells
whether the parameter was declared `symbolic`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set INTER;                                # intersections
param entr symbolic in INTER;             # entrance: must be an intersection
param exit symbolic in INTER, <> entr;    # exit: an intersection, not the entrance

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["INTER"] = ["a", "b", "c", "d", "e", "f", "g"]
ampl.param["entr"] = "a"
ampl.param["exit"] = "g"

In [5]:
# run.py
print(ampl.param["entr"].value(), ampl.param["exit"].value())
print(ampl.param["entr"].is_symbolic())

a g
True


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set INTER := a b c d e f g;
param entr := a;
param exit := g;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display entr, exit;

entr = a
exit = g

